# Experiment

This notebook demonstrates a simplified
Graph Neural Network pipeline for
K/π particle identification.

Workflow:
1. Load detector data
2. Preprocess photon information
3. Construct graphs
4. Train GNN model
5. Evaluate classification performance

In [1]:
# =========================
# 1. Setup project path
# =========================
import sys
from pathlib import Path

# 現在位置
current = Path.cwd()

# repo root を探す
while current != current.parent:
    if (current / "src").exists():
        break
    current = current.parent

# Pythonの検索パスに追加
sys.path.insert(0, str(current))

print("Project root:", current)

Project root: c:\Users\古澤叶大\gnn-particle-identification


In [2]:
# =========================
# 2. import
# =========================
import torch
from torch_geometric.loader import DataLoader
from src.data_loader import load_top_data
from src.preprocessing import clean_photons, compute_labels
from src.graph_builder import create_graph_for_event
from src.model import GNNClassifier
from src.train import train, test

C:\Users\古澤叶大\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# =========================
# 3. データ読み込み
# =========================
USE_DUMMY = True 
if USE_DUMMY:
    from src.dummy_data import generate_dummy_data
    t, chx, chy, trkp, eHx, eHz, pdg, numPhot = generate_dummy_data(200)
    print("Using dummy data")
else:
    data_path = "your_real_data.root"
    t, chx, chy, trkp, eHx, eHz, pdg, numPhot = load_top_data(data_path)
    print("Using real data")
print("Loaded events:", len(pdg))

Using dummy data
Loaded events: 200


In [4]:
# =========================
# 4. 前処理
# =========================
# 時間フィルタ
t, chx, chy = clean_photons(t, chx, chy)
# ラベル作成
labels = compute_labels(pdg)
# 無効ラベル削除
valid_idx = labels != -1
t = t[valid_idx]
chx = chx[valid_idx]
chy = chy[valid_idx]
trkp = trkp[valid_idx]
eHx = eHx[valid_idx]
eHz = eHz[valid_idx]
labels = labels[valid_idx]
print("Valid events:", len(labels))

Valid events: 200


In [5]:
# =========================
# 5. グラフ化
# =========================
graphs = []
for i in range(len(t)):
    g = create_graph_for_event(i, t, chx, chy, trkp, eHx, eHz, labels[i])
    if g is not None:
        graphs.append(g)
print("Number of graphs:", len(graphs))

Number of graphs: 200


In [6]:
# =========================
# 6. DataLoader & モデル
# =========================
loader = DataLoader(graphs, batch_size=32, shuffle=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GNNClassifier().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
print("Device:", device)

Device: cpu


In [7]:
# =========================
# 7. 学習
# =========================
num_epochs = 5
for epoch in range(num_epochs):
    loss = train(model, loader, optimizer, device)
    acc = test(model, loader, device)
    print(f"Epoch {epoch+1} | Loss: {loss:.4f} | Acc: {acc:.4f}")

Epoch 1 | Loss: 0.9678 | Acc: 0.5100
Epoch 2 | Loss: 0.7923 | Acc: 0.5350
Epoch 3 | Loss: 0.7609 | Acc: 0.5550
Epoch 4 | Loss: 0.7012 | Acc: 0.5650
Epoch 5 | Loss: 0.6726 | Acc: 0.6000


In [8]:
# =========================
# 8. 最終評価
# =========================
final_acc = test(model, loader, device)
print("Final Accuracy:", final_acc)

Final Accuracy: 0.6
